In [1]:
import pandas as pd
import numpy as np

train_orders = pd.read_csv("../data/orders.csv")
train_orders.columns = train_orders.columns.str.lower()

/var/folders/r3/5z7l6gmn0gz0cs0971yw4qpr0000gn/T/ipykernel_7592/1626360478.py:4: DtypeWarning: Columns (0: order_accepted_time, 1: driver_accepted_time, 2: picked_up_time, 3: delivered_time, 4: delivery_date) have mixed types. Specify dtype option on import or set low_memory=False.
  train_orders = pd.read_csv("../data/orders.csv")


In [2]:
positive = train_orders[['customer_id','location_number','vendor_id']].copy()
positive['target'] = 1

print(positive.head())
print(positive.shape)

  customer_id  location_number  vendor_id  target
0     KL09J9N                0         84       1
1     H5LGGFX                0         78       1
2     CYLZB6T                0          4       1
3     4YKUKYN                0        157       1
4     WDNU30K                0        160       1
(135303, 4)


In [3]:
customers = positive['customer_id'].unique()
all_vendors = train_orders['vendor_id'].unique()

In [4]:
import random

neg_samples = []

for i, cid in enumerate(customers):
    cust_data = positive[positive['customer_id'] == cid]
    
    locations = cust_data['location_number'].unique()
    ordered_vendors = cust_data['vendor_id'].unique()
    
    not_ordered = list(set(all_vendors) - set(ordered_vendors))
    
    for loc in locations:
        sampled = random.sample(not_ordered, min(3, len(not_ordered)))
        
        for v in sampled:
            neg_samples.append([cid, loc, v, 0])
    
    if i % 5000 == 0:
        print(i)

0
5000
10000
15000
20000
25000


In [5]:
negative = pd.DataFrame(
    neg_samples,
    columns=['customer_id','location_number','vendor_id','target']
)

df = pd.concat([positive, negative], ignore_index=True)

print("Final dataset:", df.shape)

Final dataset: (266226, 4)


In [6]:
train_customers = pd.read_csv("../data/train_customers.csv")
train_locations = pd.read_csv("../data/train_locations.csv")
vendors = pd.read_csv("../data/vendors.csv")

# normalize columns
train_customers.columns = train_customers.columns.str.lower()
train_locations.columns = train_locations.columns.str.lower()
vendors.columns = vendors.columns.str.lower()

In [7]:
df = df.merge(train_customers, on='customer_id', how='left')
df = df.merge(train_locations, on=['customer_id','location_number'], how='left')
df = df.merge(vendors, left_on='vendor_id', right_on='id', how='left')

print(df.shape)

(266743, 73)


In [8]:
# vendor popularity
vendor_popularity = train_orders['vendor_id'].value_counts()
df['vendor_popularity'] = df['vendor_id'].map(vendor_popularity)

# customer order count
customer_orders = train_orders.groupby('customer_id').size()
df['customer_order_count'] = df['customer_id'].map(customer_orders)

# distance
df['distance'] = np.sqrt(
    (df['latitude_x'] - df['latitude_y'])**2 +
    (df['longitude_x'] - df['longitude_y'])**2
)

In [9]:
df['gender'] = df['gender'].astype(str).str.lower().str.strip()

df = pd.get_dummies(
    df,
    columns=['location_type','vendor_category_en','gender'],
    dummy_na=True
)

In [10]:
y = df['target']

X = df.drop(columns=['target','customer_id'])
X = X.select_dtypes(exclude=['object'])
X = X.fillna(0)

print(X.shape)

(266743, 41)


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf = RandomForestClassifier(n_estimators=100, n_jobs=-1)
rf.fit(X_train, y_train)

preds = rf.predict_proba(X_val)[:,1]

print("ROC:", roc_auc_score(y_val, preds))

ROC: 0.9459338217788016


In [12]:
test_customers = pd.read_csv("../data/test_customers.csv")
test_locations = pd.read_csv("../data/test_locations.csv")

# normalize
test_customers.columns = test_customers.columns.str.lower()
test_locations.columns = test_locations.columns.str.lower()

In [13]:
all_vendors = vendors['id'].unique()

rows = []
for _, row in test_locations.iterrows():
    cid = row['customer_id']
    loc = row['location_number']
    
    for v in all_vendors:
        rows.append([cid, loc, v])

test_df = pd.DataFrame(rows, columns=['customer_id','location_number','vendor_id'])

In [14]:
test_df_original = test_df.copy()

In [15]:
test_df = test_df.merge(test_customers, on='customer_id', how='left')
test_df = test_df.merge(test_locations, on=['customer_id','location_number'], how='left')
test_df = test_df.merge(vendors, left_on='vendor_id', right_on='id', how='left')

In [16]:
test_df['key'] = (
    test_df['customer_id'] + "_" +
    test_df['location_number'].astype(str) + "_" +
    test_df['vendor_id'].astype(str)
)

In [17]:
vendor_popularity = train_orders['vendor_id'].value_counts()
test_df['vendor_popularity'] = test_df['vendor_id'].map(vendor_popularity)

customer_orders = train_orders.groupby('customer_id').size()
test_df['customer_order_count'] = test_df['customer_id'].map(customer_orders)

test_df['distance'] = np.sqrt(
    (test_df['latitude_x'] - test_df['latitude_y'])**2 +
    (test_df['longitude_x'] - test_df['longitude_y'])**2
)

In [18]:
test_df['gender'] = test_df['gender'].astype(str).str.lower().str.strip()

test_df = pd.get_dummies(
    test_df,
    columns=['location_type','vendor_category_en','gender'],
    dummy_na=True
)

In [19]:
train_cols = rf.feature_names_in_

test_df_model = test_df.reindex(columns=train_cols, fill_value=0)
test_df_model = test_df_model.apply(pd.to_numeric, errors='coerce').fillna(0)

In [20]:
preds = rf.predict_proba(test_df_model)[:,1]

In [21]:
test_df['target'] = preds

test_df_original['key'] = (
    test_df_original['customer_id'] + "_" +
    test_df_original['location_number'].astype(str) + "_" +
    test_df_original['vendor_id'].astype(str)
)

final_df = test_df_original.merge(
    test_df[['key','target']],
    on='key',
    how='left'
)

In [22]:
final_df['output'] = (
    final_df['customer_id'] + " X " +
    final_df['location_number'].astype(str) + " X " +
    final_df['vendor_id'].astype(str) + " " +
    final_df['target'].astype(str)
)

final_df[['output']].to_csv("submission.txt", index=False, header=False)